<!-- Copyright (c) 2026 João Frazão. SPDX-License-Identifier: Apache-2.0 -->
# 1. Dimensionamento rápido com a API pública

Dimensiona o exemplo `uav_gasoline_150kg.yaml` ao nível **L0** (métodos de manual: Raymer, Gudmundsson), lê o resumo, a matriz de verificação dos requisitos e o estado de projeto `design.json` com unidades.

Definições baratas (L0, sem seleção de sistemas): ~10 s. Para low-fi (XFOIL + AVL) use `level="L1"` (precisa de `avl`/`xfoil` no PATH; senão NeuralFoil + Schrenk). Política da API: `docs/API.md`.

In [ ]:
import os, tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")  # sem ecrã (CI); no Jupyter as figuras aparecem na mesma com display
import matplotlib.pyplot as plt
from acdt import api

# raiz do repositório (o notebook corre em examples/notebooks ou na raiz)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
print(api.version_info())

In [ ]:
mission = ROOT / "examples" / "uav_gasoline_150kg.yaml"
out = Path(tempfile.mkdtemp(prefix="acdt_nb1_"))
r = api.size(mission, out, level="L0", no_systems=True)
{k: r[k] for k in ("name", "feasible", "W0_kg", "We_kg", "empty_fraction", "S_m2", "span_m", "iterations")}

Avisos do pipeline e verificações que falharam (uma verificação que falha não é escondida):

In [ ]:
for i in r["issues"][:10]:
    print(f"[{i['severity']}] {i['message'][:140]}")
print("verificações em falha:", r["failed_checks"] or "nenhuma")

## Matriz de verificação

Estados: `passa`, `falha`, `no_limite`, `nao_avaliado`, `desconhecido` — não avaliado nunca conta como passa (`docs/REQUISITOS.md`).

In [ ]:
m = api.verify(mission, out)
print(m["coverage"]["por_estado"])
for row in m["rows"]:
    print(f"{row['status']:>13}  {row['id']:<28} {str(row.get('value'))[:20]:<20} {row['text'][:60]}")

## Estado de projeto (`design.json`)

Cada grandeza com unidade, nível de fidelidade e proveniência; desconhecido é `null`, nunca zero (`docs/DADOS.md`).

In [ ]:
db = api.db_load(out)
print("válido:", api.db_validate(db) == [], "| revisão", db.get("revision"))
from acdt.core import design_db as D  # interno: só para listar caminhos neste exemplo
paths = [p for p in D.flatten(db) if p.endswith("W0_kg")]
for p in paths[:3]:
    print(p, "=", api.db_get(db, p), "kg  =", round(api.db_get(db, p, "lb"), 1), "lb")

## Diagrama de restrições

A figura escrita pelo pipeline na pasta de saída:

In [ ]:
png = out / "constraint_diagram.png"
if png.is_file():
    img = plt.imread(png)
    plt.figure(figsize=(7, 5)); plt.imshow(img); plt.axis("off"); plt.show()
else:
    print("figura não produzida nesta configuração")